# 04 -- CpG Analysis

Analysis-only notebook: no model training, no re-running the CNN. Reads the
already-saved 21bp predictions from `notebooks/01_main_experiment.ipynb`
(notebook 01) and evaluates them split by CpG status, to test the hypothesis
that whatever sequence-recoverable signal exists is concentrated at CpG
sites (spontaneous deamination of methylated C -> T) and largely absent
elsewhere.

**Schema resolutions reused as-is from `notebooks/03_position_analysis.ipynb`**
(not rediscovered here):
- `results/main/{hotspot,rare}/predictions.parquet` -- columns: `position_id,
  window_size, dataset, true_label, predicted_label, probabilities`.
  `window_size` is always 21; this notebook only uses 21bp predictions.
- `data/processed/position_table.csv` has `position_id, n_distinct_subtypes,
  shannon_entropy, cluster_id, majority_subtype` -- but NOT `hotspot_flag`,
  `majority_subtype_freq`, or `is_cpg` natively.
- `is_cpg` is computed the same way notebook 03 computed it: from
  `data/processed/tp53_mutation_dataset_w21.csv` (the single combined 21bp
  instance table), via `compute_is_cpg()` in `src/data.py` -- one `is_cpg`
  value per `position_id`, not recomputed per instance, and not sourced from
  the six-ways-split `data/splits/window_21/{hotspot,rare}/*.csv` files.

**`hotspot_flag` / `cluster_table.csv` join: skipped, and here's why.**
Notebook 03 needed that join because it built one combined position table
covering both datasets. Here, `results/main/hotspot/predictions.parquet` and
`results/main/rare/predictions.parquet` are already split by dataset (each
file's own `dataset` column is constant and already confirmed correct in
notebook 01's cross-check), so re-deriving hotspot/rare membership via
`cluster_table.hotspot_flag` would be redundant -- there's no place in this
notebook a `position_id`'s hotspot/rare group is in question. Confirmed
deliberately, not skipped by oversight.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASS_TO_IDX, compute_is_cpg, load_position_table

COMBINED_W21_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv')
POSITION_TABLE_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'position_table.csv')
MAIN_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'main')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'cpg')
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = ['hotspot', 'rare']
MIN_MEANINGFUL_N = 50  # below this, a subset comparison is flagged as too small to be meaningful

print(f"Project root:       {PROJECT_ROOT}")
print(f"Combined 21bp data: {COMBINED_W21_PATH}")
print(f"Main results dir:   {MAIN_RESULTS_DIR}")
print(f"Output dir:         {RESULTS_DIR}")


Project root:       C:\Users\danya\Documents\projects\tp53_mutation_subtype
Combined 21bp data: C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w21.csv
Main results dir:   C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main
Output dir:         C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg


## Step 1: attach `is_cpg` to predictions

**Window-size scoping note (same as notebook 03):** `is_cpg` is derived from
the 21bp window's `Sequence` (centre base at index 10, next base at index
11 -- see `compute_is_cpg` in `src/data.py`), so it is inherently a
21bp-specific quantity. This notebook only ever uses 21bp predictions from
notebook 01, so that's consistent -- stated explicitly rather than left
implicit, since `is_cpg` should not be reused as-is against predictions from
a different window size without recomputing it from that window's own
`Sequence`.


In [3]:
combined_w21 = pd.read_csv(COMBINED_W21_PATH, dtype={'position_id': str})
required_cols = {'position_id', 'Sequence', 'MutationType'}
assert required_cols.issubset(combined_w21.columns), (
    f"{COMBINED_W21_PATH} is missing required columns: "
    f"{required_cols - set(combined_w21.columns)} (has: {combined_w21.columns.tolist()})"
)

# is_cpg: one flag per position_id, from that position's 21bp Sequence (identical for every
# instance row at a given position_id and window size, so `.first()` is exact, not a sample).
seq_per_position = combined_w21.groupby('position_id')['Sequence'].first()
is_cpg_per_position = pd.Series(
    compute_is_cpg(seq_per_position.values, center_idx=10),
    index=seq_per_position.index,
    name='is_cpg',
)
print(f"is_cpg computed for {len(is_cpg_per_position):,} positions "
      f"({is_cpg_per_position.sum():,} CpG-context, {(~is_cpg_per_position).sum():,} non-CpG)")

# position_id -> cluster_id map: the true independence unit for significance
# testing is the locus (cluster), not position_id -- COSMIC's per-isoform
# re-annotation means one physical locus can carry several position_ids
# (mean 14.8 per cluster; see notebooks/data_pipeline.ipynb Stage 5), so a
# test that groups by position_id alone still pools multiple non-independent
# samples from the same locus.
position_to_cluster = load_position_table(POSITION_TABLE_PATH).set_index('position_id')['cluster_id']

# is_cpg is a property of physical sequence context, so it should agree for
# every position_id sharing a cluster -- verify rather than assume, since a
# silent disagreement (e.g. from isoform-relative flanking differences right
# at the 21bp window's edge) would mean picking `.first()` per cluster below
# is arbitrary rather than exact.
_cpg_by_cluster_check = pd.DataFrame({
    'is_cpg': is_cpg_per_position,
    'cluster_id': position_to_cluster.reindex(is_cpg_per_position.index),
})
_disagreements = _cpg_by_cluster_check.groupby('cluster_id')['is_cpg'].nunique()
_n_disagree = int((_disagreements > 1).sum())
print(f"is_cpg agreement check: {_n_disagree} cluster(s) with disagreeing is_cpg "
      f"across their position_ids (out of {_disagreements.shape[0]:,} clusters)")
if _n_disagree > 0:
    print("  NOTE: is_cpg is not perfectly cluster-consistent -- .first() below "
          "picks an arbitrary position_id's value per cluster in these cases.")

predictions = {}
for name in DATASETS:
    pred_path = os.path.join(MAIN_RESULTS_DIR, name, 'predictions.parquet')
    preds = pd.read_parquet(pred_path)

    assert set(preds.columns) == {'position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities'}, (
        f"{pred_path}: unexpected columns {preds.columns.tolist()}"
    )
    assert (preds['window_size'] == 21).all(), f"{pred_path}: expected window_size == 21 everywhere"
    assert (preds['dataset'] == name).all(), f"{pred_path}: expected dataset == '{name}' everywhere"

    missing = set(preds['position_id']) - set(is_cpg_per_position.index)
    assert not missing, f"{name}: {len(missing)} position_id(s) missing from is_cpg lookup"

    preds = preds.copy()
    preds['is_cpg'] = preds['position_id'].map(is_cpg_per_position)
    preds['correct'] = preds['true_label'] == preds['predicted_label']

    predictions[name] = preds
    print(f"{name}: {len(preds):,} instance rows, {preds['position_id'].nunique():,} positions, "
          f"is_cpg attached")


is_cpg computed for 13,423 positions (1,071 CpG-context, 12,352 non-CpG)
is_cpg agreement check: 0 cluster(s) with disagreeing is_cpg across their position_ids (out of 906 clusters)
hotspot: 112,511 instance rows, 1,136 positions, is_cpg attached
rare: 2,362 instance rows, 884 positions, is_cpg attached


## Step 2: metrics per (dataset, is_cpg) subset -- instance-level

Accuracy, MCC, macro F1, and the majority-class baseline **within each
subset** (its own most common `true_label`, not the whole-dataset baseline
from `results/main/summary.csv` -- CpG and non-CpG subsets can have
different majority labels).


In [4]:
metrics_rows = []

for name in DATASETS:
    preds = predictions[name]
    for cpg_flag in (True, False):
        subset = preds[preds['is_cpg'] == cpg_flag]

        y_true = subset['true_label'].map(CLASS_TO_IDX).values
        y_pred = subset['predicted_label'].map(CLASS_TO_IDX).values

        accuracy = accuracy_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        majority_label = subset['true_label'].mode()[0]
        majority_accuracy = (subset['true_label'] == majority_label).mean()

        metrics_rows.append({
            'dataset': name,
            'is_cpg': cpg_flag,
            'accuracy': accuracy,
            'mcc': mcc,
            'macro_f1': macro_f1,
            'majority_accuracy': majority_accuracy,
            'n_instances': int(len(subset)),
            'n_unique_positions': int(subset['position_id'].nunique()),
        })

metrics_df = pd.DataFrame(metrics_rows, columns=[
    'dataset', 'is_cpg', 'accuracy', 'mcc', 'macro_f1', 'majority_accuracy',
    'n_instances', 'n_unique_positions',
])

metrics_path = os.path.join(RESULTS_DIR, 'metrics.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f"Metrics written -> {metrics_path}\n")
metrics_df


Metrics written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\metrics.csv



,dataset,is_cpg,accuracy,mcc,macro_f1,majority_accuracy,n_instances,n_unique_positions
0,hotspot,True,0.080050,0.089331,0.056735,0.857788,71555,178
1,hotspot,False,0.223313,0.086017,0.188011,0.285282,40956,958
2,rare,True,0.384615,0.000000,0.185185,0.461538,104,16
3,rare,False,0.210363,0.025044,0.192504,0.356953,2258,868


## Step 3: CpG vs non-CpG significance test -- CLUSTER (LOCUS) level, not instance-level or position_id-level

**This is the critical methodological requirement, not a stylistic choice.**
With only 906 independent genomic loci in this study (each COSMIC mutation
replicated many times over via the `Count` field, and further fragmented
across up to ~19 `position_id`s per locus by isoform re-annotation -- mean
14.8 `position_id`s per cluster), a significance test run directly on the
~60,000-114,000 instance-level rows in `predictions.parquet` would be
pseudoreplicated. Grouping by `position_id` alone is not sufficient either:
`position_id` is a (transcript isoform, CDS position) pair, not the study's
independence unit -- several `position_id`s can share one physical locus
(`cluster_id`), so a position_id-level test still pools multiple correlated
samples from the same locus as if they were independent evidence. Every test
below is computed on **one row per `cluster_id`** (mean accuracy across every
instance belonging to that locus, however many `position_id`s or raw
instances it fragments into).

**Test choice:** Mann-Whitney U (`scipy.stats.mannwhitneyu`), comparing the
distribution of per-cluster accuracy between CpG and non-CpG loci,
separately for hotspot and rare. Chosen because per-cluster accuracy is a
bounded [0,1] value derived from a small, variable number of instances per
locus (often producing exact 0s and 1s, especially for low-`n_instances`
loci) -- not something a normality assumption (as a t-test would require) is
appropriate for. Mann-Whitney U only assumes two independent samples and
tests whether one distribution is stochastically greater than the other,
which matches the comparison being made here.


In [5]:
per_position_frames = {}  # name kept for downstream compatibility; now cluster-level, not position_id-level

for name in DATASETS:
    preds = predictions[name].copy()
    preds['cluster_id'] = preds['position_id'].map(position_to_cluster)
    assert preds['cluster_id'].notna().all(), f"{name}: some position_id(s) missing from position_to_cluster"

    per_position = (
        preds.groupby('cluster_id')
        .agg(accuracy=('correct', 'mean'), is_cpg=('is_cpg', 'first'), n_position_ids=('position_id', 'nunique'))
        .reset_index()
        .rename(columns={'cluster_id': 'position_id'})  # keep column name for downstream cells
    )
    per_position_frames[name] = per_position
    print(f"{name}: {len(per_position):,} unique clusters (loci) "
          f"({per_position['is_cpg'].sum():,} CpG, {(~per_position['is_cpg']).sum():,} non-CpG), "
          f"mean {per_position['n_position_ids'].mean():.1f} position_ids/cluster")

hotspot: 70 unique clusters (loci) (12 CpG, 58 non-CpG), mean 16.2 position_ids/cluster


rare: 67 unique clusters (loci) (2 CpG, 65 non-CpG), mean 13.2 position_ids/cluster


In [6]:
significance_results = {}

for name in DATASETS:
    per_position = per_position_frames[name]
    cpg_acc = per_position.loc[per_position['is_cpg'], 'accuracy'].values
    noncpg_acc = per_position.loc[~per_position['is_cpg'], 'accuracy'].values

    statistic, pvalue = mannwhitneyu(cpg_acc, noncpg_acc, alternative='two-sided')

    significance_results[name] = {
        'level': 'cluster (locus-level, NOT position_id-level or instance-level)',
        'test': 'mannwhitneyu (two-sided)',
        'statistic': float(statistic),
        'p_value': float(pvalue),
        'n_cpg_positions': int(len(cpg_acc)),      # key name kept for downstream compatibility; counts CLUSTERS
        'n_noncpg_positions': int(len(noncpg_acc)),  # key name kept for downstream compatibility; counts CLUSTERS
        'median_accuracy_cpg': float(np.median(cpg_acc)),
        'median_accuracy_noncpg': float(np.median(noncpg_acc)),
    }

significance_path = os.path.join(RESULTS_DIR, 'significance.json')
with open(significance_path, 'w') as f:
    json.dump(significance_results, f, indent=2)

print(f"Cluster-level significance results written -> {significance_path}\n")
for name, res in significance_results.items():
    print(f"{name}: Mann-Whitney U={res['statistic']:.1f}  p={res['p_value']:.4g}  "
          f"n_cpg={res['n_cpg_positions']}  n_noncpg={res['n_noncpg_positions']}  "
          f"median_acc(CpG)={res['median_accuracy_cpg']:.3f}  median_acc(non-CpG)={res['median_accuracy_noncpg']:.3f}")


Cluster-level significance results written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\significance.json

hotspot: Mann-Whitney U=277.0  p=0.227  n_cpg=12  n_noncpg=58  median_acc(CpG)=0.000  median_acc(non-CpG)=0.000
rare: Mann-Whitney U=101.0  p=0.1473  n_cpg=2  n_noncpg=65  median_acc(CpG)=0.600  median_acc(non-CpG)=0.000


## Step 4: comparison figure

In [7]:
fig, ax = plt.subplots(figsize=(7, 5))

x_positions = np.arange(len(DATASETS))
bar_width = 0.35
colors = {True: 'tab:red', False: 'tab:gray'}
labels = {True: 'CpG', False: 'non-CpG'}

for i, cpg_flag in enumerate((True, False)):
    heights = [
        metrics_df.loc[(metrics_df['dataset'] == name) & (metrics_df['is_cpg'] == cpg_flag), 'accuracy'].iloc[0]
        for name in DATASETS
    ]
    baselines = [
        metrics_df.loc[(metrics_df['dataset'] == name) & (metrics_df['is_cpg'] == cpg_flag), 'majority_accuracy'].iloc[0]
        for name in DATASETS
    ]
    offset = (i - 0.5) * bar_width
    bars = ax.bar(x_positions + offset, heights, bar_width, label=labels[cpg_flag], color=colors[cpg_flag])

    # subset's own majority baseline as a marker on top of its bar
    for xp, base in zip(x_positions + offset, baselines):
        ax.plot([xp - bar_width / 2, xp + bar_width / 2], [base, base], color='black', linewidth=2)

ax.plot([], [], color='black', linewidth=2, label="subset's own majority baseline")
ax.set_xticks(x_positions)
ax.set_xticklabels(DATASETS)
ax.set_ylabel('Accuracy')
ax.set_title('CpG vs. non-CpG accuracy, with each subset\'s own majority baseline')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig_path = os.path.join(RESULTS_DIR, 'comparison.png')
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print(f"Figure saved -> {fig_path}")


Figure saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\comparison.png


## Final summary

In [8]:
print("=" * 100)
print("metrics.csv (instance-level accuracy/MCC/macro-F1/majority-baseline, per subset)")
print("=" * 100)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print(metrics_df.to_string(index=False))

print("\n" + "=" * 100)
print("significance.json (POSITION-LEVEL Mann-Whitney U, CpG vs non-CpG)")
print("=" * 100)
for name, res in significance_results.items():
    print(f"\n{name}:")
    for k, v in res.items():
        print(f"  {k}: {v}")

print("\n" + "=" * 100)
print("SAMPLE SIZE ADEQUACY CHECK")
print("=" * 100)
print(f"(flag threshold: n_unique_positions < {MIN_MEANINGFUL_N} in either group)")
for name in DATASETS:
    res = significance_results[name]
    for group_label, n in (('CpG', res['n_cpg_positions']), ('non-CpG', res['n_noncpg_positions'])):
        if n < MIN_MEANINGFUL_N:
            print(f"  FLAGGED: {name}/{group_label} has only {n} unique positions -- "
                  f"too few (< {MIN_MEANINGFUL_N}) for this comparison to be considered "
                  "statistically meaningful, even if the p-value looks significant. "
                  "This is a real limitation on THIS specific comparison, on top of the "
                  "~300-460 total independent-position ceiling already established for the study.")
        else:
            print(f"  OK: {name}/{group_label} has {n} unique positions (>= {MIN_MEANINGFUL_N}).")

print("\nOutputs:")
print(f"  {os.path.join(RESULTS_DIR, 'metrics.csv')}")
print(f"  {os.path.join(RESULTS_DIR, 'significance.json')}")
print(f"  {os.path.join(RESULTS_DIR, 'comparison.png')}")


metrics.csv (instance-level accuracy/MCC/macro-F1/majority-baseline, per subset)
dataset  is_cpg  accuracy      mcc  macro_f1  majority_accuracy  n_instances  n_unique_positions
hotspot    True  0.080050 0.089331  0.056735           0.857788        71555                 178
hotspot   False  0.223313 0.086017  0.188011           0.285282        40956                 958
   rare    True  0.384615 0.000000  0.185185           0.461538          104                  16
   rare   False  0.210363 0.025044  0.192504           0.356953         2258                 868

significance.json (POSITION-LEVEL Mann-Whitney U, CpG vs non-CpG)

hotspot:
  level: cluster (locus-level, NOT position_id-level or instance-level)
  test: mannwhitneyu (two-sided)
  statistic: 277.0
  p_value: 0.22703792074127027
  n_cpg_positions: 12
  n_noncpg_positions: 58
  median_accuracy_cpg: 0.0
  median_accuracy_noncpg: 0.0

rare:
  level: cluster (locus-level, NOT position_id-level or instance-level)
  test: mannwhitney